In [1]:
!pip install -q langchain-chroma langchain-openai langchain-huggingface langchain-community python-dotenv "opentelemetry-api==1.42.1" "opentelemetry-sdk==1.42.1" "huggingface-hub>=0.33.4,<2.0.0"


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.


In [2]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

/tmp/ipykernel_3389/3216454956.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [4]:
MODEL = "nvidia/nemotron-3.5-lightning"
DB_NAME = "vector_db"

In [5]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
retriever = vectorstore.as_retriever()
from langchain_openai import ChatOpenAI
from google.colab import userdata

MODEL = "nvidia/nemotron-3.5-lightning:free"

llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=userdata.get("OPENROUTER_API_KEY"),
    temperature=0,
    model=MODEL,
    extra_body={
        "reasoning": {
            "effort": "none"
        }
    }
)

In [7]:

retriever.invoke("Who is Avery?")

[]

In [8]:

llm.invoke("Who is Avery?")

AIMessage(content='"Avery" is a name that can refer to several different people or characters, depending on the context. Here are the most common references:\n\n**1. Avery as a Given Name**\nAvery is a unisex name of English origin, meaning "elf counsel" or "ruler of elves." It has become increasingly popular for both boys and girls in English-speaking countries.\n\n**2. Avery as a Surname**\nAvery is also a common English and French surname, derived from the Old French "Alvred" or the Anglo-Saxon "Aelfric."\n\n**3. Famous People Named Avery**\n*   **Avery Johnson** – Former NBA player and coach.\n*   **Avery Bradley** – Professional basketball player.\n*   **Avery Cyrus** – Social media personality and TikTok creator.\n*   **Avery Johnson** (singer) – American country music singer.\n\n**4. Fictional Characters**\n*   **Avery Jessup** – A character from the TV show *30 Rock*.\n*   **Avery Bullock** – A character from the animated series *American Dad!*.\n*   **Avery Martin** – A charac

In [9]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [12]:
from langchain_core.messages import SystemMessage, HumanMessage

In [13]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [14]:
answer_question("Who is Averi Lancaster?", [])

"Averi Lancaster is a professional associated with Insurellm. She works as a Business Development Representative (BDR) at the company. In her role, she focuses on prospecting, lead generation, and building relationships with potential clients to help grow Insurellm's business. Averi is known for her enthusiasm for the insurance technology space and her dedication to helping agencies and carriers find the right solutions through Insurellm's platform."

In [16]:
import gradio as gr
gr.ChatInterface(answer_question).launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://596a0c714a1e2d214a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
